## Event-level operating curves + global threshold selection (LOPOCV)
Objective: max EVENT sensitivity subject to a patient-averaged FA/h budget.

In [ ]:
import warnings

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

ev = pd.read_csv("../results/event_curves.csv")
ev = ev[np.isfinite(ev["threshold"])].copy()

# Threshold-free event-level AUPRC per model (mean across patients), used in the
# summary table below. AUROC is omitted: event-level FPR has no true-negative-event
# count, so it is not well-defined.
event_metrics = pd.read_csv("../results/event_metrics.csv")
auprc_by_model = event_metrics.groupby("model")["event_auprc"].mean()

models = sorted(ev["model"].unique())  # every model present (logistic, random_forest, xgboost, ...)
FA_BUDGET = 0.06                # tolerable false ALARMS / hour -- event-level, tune this
LEGEND_ANCHOR = (1.15, 0.5)     # all legends: centered just outside each subplot's right edge


def interp_mean_event_curve(model_df):
    """Interpolate each patient's threshold-indexed metrics onto a shared grid, then
    average across patients -- the basis for choosing one global threshold. Returns
    mean event sensitivity, FA/h, latency, precision, and F1.

    Precision and F1 are seizure-centric (detected = TP, false-alarm events = FP,
    missed seizures = FN), computed per patient from the count columns. Latency,
    precision, and F1 are averaged only where defined per patient (a fold
    contributes NaN outside its valid threshold range and is skipped by nanmean),
    so non-detecting folds don't drag the mean."""
    patients = sorted(model_df["patient"].unique())
    grid = np.unique(np.clip(model_df["threshold"].to_numpy(), 0.001, 0.999))
    sens = np.empty((len(patients), grid.size))
    fa = np.empty((len(patients), grid.size))
    lat = np.full((len(patients), grid.size), np.nan)
    prec = np.full((len(patients), grid.size), np.nan)
    f1 = np.full((len(patients), grid.size), np.nan)

    def masked_interp(t, vals):
        """Interpolate onto grid over finite values only; NaN outside their range."""
        ok = np.isfinite(vals)
        out = np.full(grid.size, np.nan)
        if ok.any():
            out = np.interp(grid, t[ok], vals[ok])
            out[(grid < t[ok].min()) | (grid > t[ok].max())] = np.nan
        return out

    for i, p in enumerate(patients):
        g = model_df[model_df["patient"] == p].sort_values("threshold")
        t = np.clip(g["threshold"].to_numpy(), 0.0, 1.0)
        sens[i] = np.interp(grid, t, g["event_sensitivity"].to_numpy())
        fa[i] = np.interp(grid, t, g["fa_per_hour"].to_numpy())
        nd = g["n_detected"].to_numpy().astype(float)
        nfa = g["n_false_alarms"].to_numpy().astype(float)
        nsz = g["n_seizures"].to_numpy().astype(float)
        with np.errstate(invalid="ignore", divide="ignore"):
            row_prec = np.where(nd + nfa > 0, nd / (nd + nfa), np.nan)
            row_f1 = np.where(nd + nsz + nfa > 0, 2 * nd / (nd + nsz + nfa), np.nan)
        lat[i] = masked_interp(t, g["median_latency"].to_numpy())
        prec[i] = masked_interp(t, row_prec)
        f1[i] = masked_interp(t, row_f1)
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", category=RuntimeWarning)  # all-NaN columns -> NaN
        mlat = np.nanmean(lat, axis=0)
        mprec = np.nanmean(prec, axis=0)
        mf1 = np.nanmean(f1, axis=0)
    return grid, sens.mean(0), fa.mean(0), mlat, mprec, mf1, patients


def froc_envelope_mean(model_df, x_col, y_col, n_grid=120):
    """Mean FROC from per-patient Pareto envelopes.

    Each patient's operating points are reduced to the upper-left envelope -- the
    max achievable sensitivity at FA/h <= x (cumulative max of sensitivity over
    increasing FA/h, plus a (0, 0) 'never fire' anchor) -- which is monotonic by
    construction, unlike the raw threshold trace. Envelopes are step-evaluated on
    a shared log-spaced FA/h grid and averaged vertically across patients.
    Returns (grid, mean_sens, per_patient_sens, patients)."""
    patients = sorted(model_df["patient"].unique())
    x_all = model_df[x_col].to_numpy()
    pos = x_all[x_all > 0]
    x_min = pos.min() if pos.size else 1.0
    grid = np.concatenate([[0.0], np.logspace(np.log10(x_min), np.log10(x_all.max()), n_grid)])
    per_patient = []
    for p in patients:
        g = model_df[model_df["patient"] == p]
        x = np.append(g[x_col].to_numpy(), 0.0)   # 'never fire' point: 0 FA/h, 0 sensitivity
        y = np.append(g[y_col].to_numpy(), 0.0)
        order = np.argsort(x, kind="stable")
        xs, ys = x[order], np.maximum.accumulate(y[order])
        idx = np.clip(np.searchsorted(xs, grid, side="right") - 1, 0, len(ys) - 1)
        per_patient.append(ys[idx])
    per_patient = np.array(per_patient)
    return grid, per_patient.mean(0), per_patient, patients


def select_operating_point(mean_sens, mean_fa, *required_finite, budget):
    """Bottom-panel operating-point selection over rows with no NaN.

    Restricts to indices where all `required_finite` arrays are finite, then picks
    the highest-sensitivity index whose mean FA/h <= budget. If no row meets the
    budget, falls back to the row with the smallest mean FA/h among valid rows."""
    valid = np.ones(mean_fa.size, dtype=bool)
    for m in required_finite:
        valid &= np.isfinite(m)
    if not valid.any():
        return None
    feasible = valid & (mean_fa <= budget)
    if feasible.any():
        return int(np.where(feasible, mean_sens, -np.inf).argmax())
    return int(np.where(valid, mean_fa, np.inf).argmin())  # smallest FA/h fallback


def normalized_auc(y, x):
    """Trapezoidal area under y vs monotone x, normalized by the x-range -> [0, 1].

    For an FROC envelope (sensitivity vs FA/h, sensitivity in [0, 1]), this is the
    mean achievable sensitivity across the swept FA/h range."""
    x_range = x[-1] - x[0]
    if x_range <= 0:
        return float("nan")
    return float(np.sum((y[:-1] + y[1:]) / 2 * np.diff(x)) / x_range)


def target_status(floor_fa, budget):
    """One-row status comparing the model's smallest reachable mean FA/h (the floor)
    to the target budget. '>>' uses a 2x cutoff."""
    if not np.isfinite(floor_fa):
        return "n/a"
    if floor_fa <= budget:
        return "at target"
    if floor_fa < 2 * budget:
        return "floor > target"
    return "floor >> target"


fig, axes = plt.subplots(2, len(models), figsize=(7.5 * len(models), 9), layout="constrained", squeeze=False)
summary_rows = []
for j, model in enumerate(models):
    mdf = ev[ev["model"] == model]
    grid, msens, mfa, mlat, mprec, mf1, patients = interp_mean_event_curve(mdf)
    # Bottom panel uses NaN-free rows only; t* picked at FA_BUDGET with fallback to min FA/h.
    valid = np.isfinite(mlat) & np.isfinite(mprec) & np.isfinite(mf1)
    idx = select_operating_point(msens, mfa, mlat, mprec, mf1, budget=FA_BUDGET)
    # mathtext-safe model name for a bold title (\_ escapes underscores so e.g.
    # 'random_forest' doesn't render with 'forest' as a subscript). The escape is
    # built outside the f-string because Py3.11 forbids backslashes in expressions.
    escaped_model = model.replace("_", r"\_")
    model_bold = rf"$\mathbf{{{escaped_model}}}$"

    # Top: event-level FROC via per-patient Pareto envelopes (max sensitivity at
    # FA/h <= x), step-evaluated on a shared log-spaced grid, averaged vertically.
    ax = axes[0, j]
    fa_grid, mean_env, env_pp, env_patients = froc_envelope_mean(mdf, "fa_per_hour", "event_sensitivity")
    # AUC under each envelope, normalized to the swept FA/h range -> [0, 1].
    pp_aucs = [normalized_auc(row, fa_grid) for row in env_pp]
    mean_auc = normalized_auc(mean_env, fa_grid)
    colors = plt.get_cmap("tab10")(np.arange(len(env_pp)) % 10)
    for row, c, p, auc in zip(env_pp, colors, env_patients, pp_aucs):
        ax.plot(fa_grid, row, color=c, lw=1.0, alpha=0.4, zorder=1,
                label=f"patient {p}  AUC={auc:.3f}")
    ax.plot(fa_grid, mean_env, color="k", lw=2.4,
            label=f"mean envelope  AUC={mean_auc:.3f}", zorder=3)
    ax.axvline(FA_BUDGET, color="C3", ls="--", lw=1, label=f"budget = {FA_BUDGET:g} FA/h")
    ax.set_xscale("symlog", linthresh=1)
    ax.set_xlabel("false alarms / hour")
    ax.set_ylabel("event sensitivity")
    ax.set_ylim(-0.02, 1.02)
    ax.set_title(f"{model_bold}\nevent FROC envelope (colored = per patient)")
    leg = ax.legend(loc="center left", bbox_to_anchor=LEGEND_ANCHOR, fontsize=7)
    for lh in leg.legend_handles:
        lh.set_alpha(1.0)  # plotted lines stay faint, but legend swatches read crisply

    # Bottom: threshold sweep restricted to NaN-free rows; t* marked with FA_BUDGET
    # selection (falls back to the smallest mean FA/h if 0.06 isn't reachable).
    ax = axes[1, j]
    grid_v, msens_v, mfa_v = grid[valid], msens[valid], mfa[valid]
    ax.plot(grid_v, msens_v, color="C0", label="mean event sensitivity")
    ax.set_xlabel("threshold")
    ax.set_ylabel("mean event sensitivity", color="C0")
    ax.set_ylim(-0.02, 1.02)
    ax.tick_params(axis="y", labelcolor="C0")
    ax2 = ax.twinx()
    ax2.plot(grid_v, mfa_v, color="C1", label="mean FA/h")
    ax2.axhline(FA_BUDGET, color="C3", ls="--", lw=1)
    ax2.set_ylabel("mean FA/h", color="C1")
    ax2.set_yscale("symlog", linthresh=1)
    ax2.tick_params(axis="y", labelcolor="C1")
    # Combined legend (blue on ax, orange on the twin ax2), outside on the right.
    lines1, labels1 = ax.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax2.legend(lines1 + lines2, labels1 + labels2, loc="center left",
               bbox_to_anchor=LEGEND_ANCHOR, fontsize=8)
    if idx is not None:
        t_star, s_star, f_star, l_star = grid[idx], msens[idx], mfa[idx], mlat[idx]
        ax.axvline(t_star, color="k", ls=":", lw=1.2)
        lat_txt = "n/a" if not np.isfinite(l_star) else f"{l_star:.1f}s"
        fallback_tag = "  [fallback: min FA/h]" if f_star > FA_BUDGET else ""
        ax.set_title(
            f"{model} — t*={t_star:.4g} (sens={s_star:.2f}, {f_star:.3g} FA/h, lat={lat_txt}){fallback_tag}"
        )
    else:
        ax.set_title(f"{model} — no NaN-free operating point")

    # One row per model for the markdown summary: metrics at the bottom-panel
    # operating point, AUPRC from event_metrics, and a status vs. the FA/h target.
    floor_fa = float(mfa[valid].min()) if valid.any() else float("nan")
    if idx is not None:
        fa_at_op = float(mfa[idx])
        sens_at_op = float(msens[idx])
        prec_at_op = float(mprec[idx]) if np.isfinite(mprec[idx]) else float("nan")
        lat_at_op = float(mlat[idx]) if np.isfinite(mlat[idx]) else float("nan")
    else:
        fa_at_op = sens_at_op = prec_at_op = lat_at_op = float("nan")
    summary_rows.append({
        "model": model,
        "AUPRC": round(float(auprc_by_model.get(model, np.nan)), 4),
        "FA/h": round(fa_at_op, 4),
        "sens": round(sens_at_op, 3),
        "precision": round(prec_at_op, 3),
        "latency": round(lat_at_op, 1),
        "FA/h at target (0.06/h)?": target_status(floor_fa, FA_BUDGET),
    })

plt.show()
fig.savefig("../figures/baseline_curves.png", dpi=300, bbox_inches="tight")

# Markdown summary -- one row per model at the bottom-panel operating point.
summary_df = pd.DataFrame(summary_rows)
print(summary_df.to_markdown(index=False))
